# FlyRank ML Capstone — Refresh / Content Opportunity Scoring

**Author:** Soyeb Ahmed  
**Lane:** Refresh / Content Opportunity Scoring  
**Question:** Can prior search and content signals rank pages that are more likely to decline in the next 30-day comparison window?

> This notebook intentionally uses a leakage-safe feature boundary. The label is defined by the latest 30-day window, so latest-window and 90-day measurements that overlap that label are excluded from the model.

## 1. Imports and reproducibility
Random seed: **42**.

In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score, accuracy_score

SEED = 42
np.random.seed(SEED)

# Works from the repository or from the notebook directory inside the repo.
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    p = candidate / "data" / "raw" / "content_refresh_anonymized.csv"
    if p.exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from the repository so data/raw/content_refresh_anonymized.csv is available.")

RAW_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(RAW_PATH)
print("Dataset:", df.shape)
df.head(3)

## 2. Research question and label

**Decision:** which content items should a human reviewer inspect first?

**Label:** `1` when `trend_direction == "down"`, meaning latest-30-day impressions are more than 20% below the preceding-30-day impressions.

This is a retrospective decline-ranking task. It is **not** a causal estimate of content refresh impact.

In [ ]:
df["label"] = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
print("Rows:", len(df))
print("Positive rows:", int(df["label"].sum()))
print("Positive rate:", round(df["label"].mean(), 3))

## 3. Leakage audit and feature boundary

The following are **excluded** because they contain the target period or the label itself:

- `trend_direction`, `trend_pct`
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`
- 90-day totals/rates that include the latest 30 days
- current freshness fields such as `days_since_last_update`
- `content_id`, `client_id` as model features

The model uses the **previous 30-day window** plus static content/keyword metadata.

In [ ]:
df["prior_ctr"] = np.where(
    df["impressions_prev_30d"] > 0,
    df["clicks_prev_30d"] / df["impressions_prev_30d"] * 100,
    0,
)
df["prior_session_rate"] = np.where(
    df["impressions_prev_30d"] > 0,
    df["sessions_prev_30d"] / df["impressions_prev_30d"] * 100,
    0,
)
df["content_age_at_start"] = np.maximum(
    pd.to_numeric(df["content_age_days"], errors="coerce").fillna(0) - 30, 0
)

numeric = [
    "competition","cpc","word_count","char_count",
    "impressions_prev_30d","clicks_prev_30d","sessions_prev_30d",
    "prior_ctr","prior_session_rate","content_age_at_start"
]
categorical = ["competition_level","content_type","main_intent","word_count_tier"]

for c in numeric:
    df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0)
for c in categorical:
    df[c] = df[c].fillna("unknown").astype(str)

Xn = df[numeric].copy()
Xn["log_search_volume"] = np.log1p(pd.to_numeric(df["search_volume"], errors="coerce").fillna(0))
for c in ["impressions_prev_30d","clicks_prev_30d","sessions_prev_30d"]:
    Xn["log_" + c] = np.log1p(Xn[c])
Xn = Xn.drop(columns=["impressions_prev_30d","clicks_prev_30d","sessions_prev_30d"])
Xcat = pd.get_dummies(df[categorical], prefix=categorical, dtype=float)
X = pd.concat([Xn.reset_index(drop=True), Xcat.reset_index(drop=True)], axis=1)
y = df["label"].astype(int).reset_index(drop=True)

print("Feature count:", X.shape[1])
print("Excluded target-period fields are not in X.")

## 4. Client-holdout validation

The same pseudonymous client must not appear in both train and test. This tests whether the ranking generalizes across clients rather than memorizing client-specific patterns.

In [ ]:
clients = df["client_id"].fillna("unknown").astype(str).reset_index(drop=True)
rng = np.random.default_rng(SEED)
unique_clients = clients.drop_duplicates().to_numpy()
test_clients = set(rng.permutation(unique_clients)[:max(1, int(round(len(unique_clients)*0.20)))])
test_mask = clients.isin(test_clients).to_numpy()

train_idx = np.where(~test_mask)[0]
test_idx = np.where(test_mask)[0]

print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Train classes:", y.iloc[train_idx].value_counts(normalize=True).to_dict())
print("Test classes:", y.iloc[test_idx].value_counts(normalize=True).to_dict())

## 5. Transparent baseline

The baseline is intentionally simple:

- 55% prior-impression percentile
- 25% prior-session percentile
- 15% inverse word-count percentile
- 5% competition percentile

In [ ]:
def pct(s):
    return pd.Series(s).rank(method="average", pct=True).to_numpy()

def precision_at_k(y_true, scores, k):
    idx = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[idx].mean())

baseline = (
    0.55 * pct(np.log1p(df["impressions_prev_30d"]))
    + 0.25 * pct(np.log1p(df["sessions_prev_30d"]))
    + 0.15 * (1 - pct(df["word_count"].fillna(0)))
    + 0.05 * pct(df["competition"].fillna(0))
)

baseline_metrics = {
    "roc_auc": roc_auc_score(y.iloc[test_idx], baseline[test_idx]),
    "average_precision": average_precision_score(y.iloc[test_idx], baseline[test_idx]),
    "precision_at_20": precision_at_k(y.iloc[test_idx], baseline[test_idx], 20),
    "precision_at_50": precision_at_k(y.iloc[test_idx], baseline[test_idx], 50),
    "precision_at_100": precision_at_k(y.iloc[test_idx], baseline[test_idx], 100),
}
baseline_metrics

## 6. Model comparison

In [ ]:
models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED))
    ]),
    "decision_tree": DecisionTreeClassifier(
        class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=SEED
    ),
    "random_forest": RandomForestClassifier(
        class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
        n_estimators=200, n_jobs=-1, random_state=SEED
    ),
}

def metric_row(y_true, p):
    return {
        "accuracy": accuracy_score(y_true, p >= 0.5),
        "precision": precision_score(y_true, p >= 0.5, zero_division=0),
        "recall": recall_score(y_true, p >= 0.5, zero_division=0),
        "f1": f1_score(y_true, p >= 0.5, zero_division=0),
        "roc_auc": roc_auc_score(y_true, p),
        "average_precision": average_precision_score(y_true, p),
        "precision_at_20": precision_at_k(y_true, p, 20),
        "precision_at_50": precision_at_k(y_true, p, 50),
        "precision_at_100": precision_at_k(y_true, p, 100),
    }

results = {}
for name, model in models.items():
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    p = model.predict_proba(X.iloc[test_idx])[:, 1]
    results[name] = metric_row(y.iloc[test_idx], p)

table = pd.DataFrame(results).T
table.loc["baseline_rules"] = {
    "accuracy": np.nan, "precision": np.nan, "recall": np.nan, "f1": np.nan,
    **baseline_metrics
}
table.round(3)

## 7. Select the ranking model

The operational metric is **Precision@50** because the first 50 recommendations are the most actionable review queue. Random forest is selected on that metric.

In [ ]:
best_name = max(results, key=lambda n: (results[n]["precision_at_50"], results[n]["average_precision"], results[n]["roc_auc"]))
best_name, results[best_name]

## 8. Feature importance and interpretation

In [ ]:
rf = RandomForestClassifier(
    class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
    n_estimators=200, n_jobs=-1, random_state=SEED
).fit(X, y)

feature_importance = (
    pd.DataFrame({"feature": X.columns, "importance": rf.feature_importances_})
    .sort_values("importance", ascending=False)
)
feature_importance.head(15)

**Interpretation:** the strongest signals are mostly prior demand/traffic and content-age/depth measures. These are associations useful for prioritization; they are not claims that changing a feature will cause a ranking improvement.

## 9. Ranked recommendations

In [ ]:
full_prob = rf.predict_proba(X)[:, 1]
final_score = 0.75 * full_prob + 0.25 * baseline

queue = df[["content_id","client_id"]].copy()
queue["refresh_score"] = final_score * 100
queue["model_probability"] = full_prob
queue["baseline_score"] = baseline
queue["observed_decline"] = y
queue["prior_impressions"] = df["impressions_prev_30d"]
queue["prior_sessions"] = df["sessions_prev_30d"]
queue["word_count"] = df["word_count"]
queue["competition_level"] = df["competition_level"]
queue["content_type"] = df["content_type"]
queue["main_intent"] = df["main_intent"]
queue["content_age_at_start"] = df["content_age_at_start"]

def reason_codes(r):
    reasons = []
    if r.prior_impressions >= 500: reasons.append("prior_search_demand")
    if r.prior_sessions >= 10: reasons.append("prior_traffic")
    if pd.notna(r.word_count) and r.word_count > 0 and r.word_count < 1200 and r.prior_impressions >= 250:
        reasons.append("thin_content")
    if str(r.competition_level) == "HIGH": reasons.append("high_competition")
    if r.content_age_at_start >= 180: reasons.append("mature_content")
    if r.model_probability >= 0.65: reasons.append("model_decline_risk")
    return "|".join(reasons or ["general_review"])

queue["reason_codes"] = queue.apply(reason_codes, axis=1)

def action(r):
    reasons = set(r.reason_codes.split("|"))
    if "thin_content" in reasons and "model_decline_risk" in reasons:
        return "expand_and_refresh"
    if "model_decline_risk" in reasons and "prior_search_demand" in reasons:
        return "refresh"
    if "high_competition" in reasons and "prior_search_demand" in reasons:
        return "review_competitive_position"
    return "monitor"

queue["suggested_action"] = queue.apply(action, axis=1)
queue["confidence"] = np.select(
    [queue.refresh_score >= 70, queue.refresh_score >= 50],
    ["high", "medium"], default="low"
)
queue = queue.sort_values("refresh_score", ascending=False).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue)+1)

queue.head(20)[[
    "rank","refresh_score","model_probability","suggested_action",
    "reason_codes","prior_impressions","prior_sessions","word_count"
]]

## 10. Error/quality checks

The top-ranked queue is retrospective: we can inspect how often the observed decline label appears in the top K, but this does not turn the score into a causal or guaranteed forecast.

In [ ]:
for k in [20, 50, 100, 500]:
    print(f"Top-{k} observed decline rate: {queue.head(k)['observed_decline'].mean():.3f}")

print("\nBase rate:", round(y.mean(), 3))
print("RF Precision@50:", round(results["random_forest"]["precision_at_50"], 3))
print("Baseline Precision@50:", round(baseline_metrics["precision_at_50"], 3))
print("Lift:", round(results["random_forest"]["precision_at_50"] / baseline_metrics["precision_at_50"], 2), "x")

## 11. Reproducibility and public-safety checklist

- [x] Client-holdout split
- [x] Fixed random seed
- [x] Label-derived fields excluded
- [x] Latest-window and overlapping 90-day features excluded
- [x] Pseudonymous IDs used only for grouping, not as model features
- [x] No private client names, domains, URLs, queries, or credentials
- [x] Claims framed as observed/measured/directional/decision-support
- [x] No claim about Google's algorithm
- [x] Starter-data scope disclosed

### ML-12 — employer-facing summary

**5-minute demo:** explain the decision, show the leakage audit, compare baseline vs random forest, show the top-ranked queue, and close with limitations.

**Social-post cut:** Built a leakage-aware content-refresh ranking model on anonymized search data, using client-holdout validation and a transparent baseline. The final random forest reached Precision@50 0.76 on the holdout versus 0.32 for the baseline.

**Employer-facing summary:** I built an explainable content-refresh prioritization workflow that turns historical search signals into a ranked review queue. I focused on leakage control, grouped validation, baseline comparison, and decision-support recommendations rather than claiming to predict Google's algorithm.